# interp-engine on Colab — eager backend

**Arrived from the diagram's Notebook button?** The snippet you clicked is on your clipboard.
Run the install cell, then paste it into the last cell and run that.

[`interp-engine`](https://github.com/decoderesearch/interp-engine) reads activations out of a
transformer at 34 standardized points — `resid_post.10`, `mlp_act.3`, `z.7` — and an address means
the same tensor on every architecture it supports. This notebook is the runnable half of the point
diagram at **[interp-engine.org](https://interp-engine.org)**, which is the cheat sheet for where
those points sit and what other stacks call them.

Every card on the diagram has its own URL, so the one this snippet came from can be reopened or
sent to someone: `https://interp-engine.org/?arch=Qwen3ForCausalLM&point=resid_post.2` is
`resid_post` at layer 2 on Qwen 3, and `?vs=` beside it draws a second architecture to compare
against. The same table as markdown, with which backend serves what, is in
[SUPPORTED_POINTS.md](https://github.com/decoderesearch/interp-engine/blob/main/docs/SUPPORTED_POINTS.md).

## Pick a runtime

**Runtime → Change runtime type → T4 GPU**, then Save. The eager backend is the one that runs
*without* CUDA, so a CPU runtime will finish rather than fail — but a multi-billion-parameter
checkpoint in fp32 on two vCPUs is a long wait per forward pass, and Colab's free GPU costs nothing.
Small checkpoints (`gpt2`, `Qwen/Qwen3-0.6B`) are genuinely fine on CPU.

If the snippet's checkpoint does not fit, edit its `load_model` line rather than the point:
`dtype="float16"` halves the weights, and any smaller checkpoint of the same architecture holds the
same points at the same addresses. Layer indices are the one thing that does not carry over — a
28-layer model has no `resid_post.40`.

This backend serves all 34 points, including the six vLLM refuses, and it is what the attention
snippets need: `attn_scores` and `attn_probs` are rebuilt from the real softmax, which requires
loading with `attn_implementation="eager"` — the eager backend does that for you. Serving a model
rather than inspecting one? The card's `vllm` tab is the same snippet with one argument changed, and
[the vLLM notebook](https://colab.research.google.com/github/decoderesearch/interp-engine/blob/main/notebooks/interp_engine_vllm.ipynb)
installs that backend instead.

In [ ]:
# A minute or so: torch is already on Colab, so this is transformers and the engine itself.
# No vLLM and no CUDA wheels -- that is what the [vllm] extra adds, and what makes it slow.
!pip install -q interp-engine

In [ ]:
# Gated checkpoints -- Gemma and Llama, among others -- need a Hugging Face token on an
# account that has accepted the model's terms. Keep it in Colab's Secrets (the key icon in
# the left sidebar) as HF_TOKEN with "Notebook access" on, and uncomment these two lines.
#
# Not as a literal in the cell: a notebook is the thing you share, and a pasted token is
# what leaks with it.

# from google.colab import userdata
# from huggingface_hub import login; login(userdata.get("HF_TOKEN"))

In [ ]:
# Paste the snippet here (Ctrl+V, or Cmd+V on a Mac), then run this cell.
#
# The diagram's Notebook button put it on your clipboard on the way in. If the clipboard
# turns out to be empty -- some browsers refuse the write outright -- reopen the point at
# interp-engine.org and press Copy beside the tabs.
#
# The first run downloads the weights, so it is slower than every run after it in the same
# session.